# EMIPredict AI — Intelligent Financial Risk Assessment Platform

### FinTech / Banking Capstone
**EMI Eligibility Classification + Maximum EMI Regression**

This Google Colab notebook implements the complete machine-learning pipeline:

1. Load the real EMI dataset, with a synthetic-data fallback
2. Data quality checks and cleaning
3. Exploratory Data Analysis (EDA)
4. Feature engineering: ratios, affordability/risk scores, and categorical encodings
5. **Classification:** Logistic Regression, Random Forest, XGBoost, Gradient Boosting
6. **Regression:** Linear Regression, Random Forest, XGBoost, Gradient Boosting
7. MLflow experiment tracking and model registry
8. Best-model selection
9. Export deployable `.joblib` pipelines for Streamlit

> **Important:** The notebook keeps preprocessing inside each sklearn Pipeline, so the exported models can accept raw dataframe columns and perform the same transformations used during training.

In [ ]:
# Cell 1 — Install dependencies
!pip -q install -U mlflow xgboost joblib seaborn

print("Dependencies installed. If Colab asks for a runtime restart, restart once and continue from Cell 2.")

In [ ]:
# Cell 2 — Imports and configuration
import os
import warnings
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from xgboost import XGBClassifier, XGBRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

BASE_DIR = Path("/content/EMIPredict_AI")
MODELS_DIR = BASE_DIR / "models"
REPORTS_DIR = BASE_DIR / "reports"
ARTIFACTS_DIR = BASE_DIR / "artifacts"

for d in [BASE_DIR, MODELS_DIR, REPORTS_DIR, ARTIFACTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# For a normal Colab session, 150K rows gives a practical runtime.
# Set TRAIN_SAMPLE = None if you want to train on every available row.
TRAIN_SAMPLE = 150_000

print("Project directory:", BASE_DIR)
print("Models directory:", MODELS_DIR)

## 1. Data Loading

The notebook first looks for the uploaded real dataset. In Colab, you can also upload a CSV manually if the file is not already present.

If no dataset is found, a **schema-accurate synthetic dataset** is generated with 400,000 records so the notebook remains runnable end-to-end.

In [ ]:
# Cell 3 — Locate/upload the real dataset
# In Google Colab, /mnt/data is normally not available.
# This cell first checks /content, then opens the Colab uploader.

import os
from pathlib import Path

candidate_paths = [
    "/content/emi_prediction_dataset (1).csv",
    "/content/emi_prediction_dataset.csv",
    "/content/EMIPredict_AI/emi_prediction_dataset.csv",
]

dataset_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if dataset_path is None:
    try:
        from google.colab import files
        print("Please upload your EMI CSV file.")
        uploaded = files.upload()
        csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("No CSV file was uploaded.")
        dataset_path = csv_files[0]
    except ImportError:
        raise FileNotFoundError(
            "Dataset not found. In Colab, upload the CSV using the Files panel."
        )

print("Using dataset:", dataset_path)
df = pd.read_csv(dataset_path, low_memory=False)
print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
# Cell 4 — Synthetic 400K fallback
def make_synthetic_emi_data(n=400_000, seed=42):
    rng = np.random.default_rng(seed)

    age = rng.integers(21, 61, n)
    gender = rng.choice(["Male", "Female"], n)
    marital_status = rng.choice(["Single", "Married"], n, p=[0.42, 0.58])
    education = rng.choice(["High School", "Graduate", "Post Graduate"], n, p=[0.20, 0.58, 0.22])
    monthly_salary = np.maximum(rng.normal(55_000, 25_000, n), 12_000)
    employment_type = rng.choice(["Salaried", "Self-Employed", "Contract"], n, p=[0.70, 0.20, 0.10])
    years_of_employment = np.clip(rng.normal(6, 4, n), 0, 30)
    company_type = rng.choice(["Private", "Government", "Startup", "Public"], n)
    house_type = rng.choice(["Owned", "Rented", "Family"], n, p=[0.35, 0.50, 0.15])
    monthly_rent = np.where(house_type == "Rented", np.maximum(rng.normal(12_000, 5_000, n), 2_000), 0)
    family_size = rng.integers(1, 7, n)
    dependents = np.minimum(rng.integers(0, 5, n), family_size - 1)
    school_fees = np.maximum(rng.normal(4_000, 3_000, n), 0)
    college_fees = np.maximum(rng.normal(3_000, 5_000, n), 0)
    travel_expenses = np.maximum(rng.normal(4_000, 2_500, n), 500)
    groceries_utilities = np.maximum(rng.normal(11_000, 4_000, n), 2_000)
    other_monthly_expenses = np.maximum(rng.normal(5_000, 3_000, n), 0)
    existing_loans = rng.choice(["Yes", "No"], n, p=[0.38, 0.62])
    current_emi_amount = np.where(existing_loans == "Yes", np.maximum(rng.normal(10_000, 7_000, n), 1_000), 0)
    credit_score = np.clip(rng.normal(700, 65, n), 300, 850)
    bank_balance = np.maximum(rng.normal(150_000, 100_000, n), 2_000)
    emergency_fund = np.maximum(rng.normal(100_000, 70_000, n), 0)
    emi_scenario = rng.choice(["New Loan", "Home Improvement", "Education", "Vehicle", "Personal"], n)
    requested_amount = np.maximum(rng.normal(600_000, 350_000, n), 50_000)
    requested_tenure = rng.choice([12, 18, 24, 36, 48, 60, 72], n)

    monthly_expenses = (
        monthly_rent + school_fees + college_fees + travel_expenses +
        groceries_utilities + other_monthly_expenses + current_emi_amount
    )
    disposable = monthly_salary - monthly_expenses
    affordability = np.maximum(disposable * 0.40, 500)
    risk_score = (
        0.003 * (700 - credit_score)
        + 0.25 * (current_emi_amount / np.maximum(monthly_salary, 1))
        + 0.15 * (monthly_expenses / np.maximum(monthly_salary, 1))
        - 0.0000015 * bank_balance
        - 0.0000010 * emergency_fund
    )
    logit = (
        1.8
        + 0.006 * (credit_score - 650)
        + 0.000008 * monthly_salary
        - 0.000010 * monthly_expenses
        - 1.4 * (existing_loans == "Yes")
        - 2.0 * np.maximum(risk_score, 0)
    )
    prob = 1 / (1 + np.exp(-np.clip(logit, -30, 30)))
    eligible = rng.random(n) < prob
    emi_eligibility = np.where(eligible, "Eligible", "Not_Eligible")

    max_monthly_emi = np.clip(
        affordability
        + 0.18 * np.maximum(credit_score - 650, 0)
        + 0.00003 * bank_balance
        + rng.normal(0, 1_000, n),
        500, None
    )

    return pd.DataFrame({
        "age": age,
        "gender": gender,
        "marital_status": marital_status,
        "education": education,
        "monthly_salary": monthly_salary,
        "employment_type": employment_type,
        "years_of_employment": years_of_employment,
        "company_type": company_type,
        "house_type": house_type,
        "monthly_rent": monthly_rent,
        "family_size": family_size,
        "dependents": dependents,
        "school_fees": school_fees,
        "college_fees": college_fees,
        "travel_expenses": travel_expenses,
        "groceries_utilities": groceries_utilities,
        "other_monthly_expenses": other_monthly_expenses,
        "existing_loans": existing_loans,
        "current_emi_amount": current_emi_amount,
        "credit_score": credit_score,
        "bank_balance": bank_balance,
        "emergency_fund": emergency_fund,
        "emi_scenario": emi_scenario,
        "requested_amount": requested_amount,
        "requested_tenure": requested_tenure,
        "emi_eligibility": emi_eligibility,
        "max_monthly_emi": max_monthly_emi
    })

if df is None:
    df = make_synthetic_emi_data()

print("Final dataset shape:", df.shape)
display(df.head())

## 2. Data Quality Checks & Cleaning

The real CSV can contain numeric columns stored as strings. We convert known numeric fields safely, normalize categorical text, remove duplicate rows, and handle missing values through the model preprocessing pipelines.

The target columns are:
- **Classification:** `emi_eligibility`
- **Regression:** `max_monthly_emi`

In [ ]:
# Cell 5 — Data quality report
print("Shape:", df.shape)
print("\nDuplicate rows:", df.duplicated().sum())
print("\nData types:")
display(df.dtypes.to_frame("dtype"))

quality = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
}).sort_values("missing_pct", ascending=False)

display(quality)

print("\nTarget distribution:")
if "emi_eligibility" in df:
    display(df["emi_eligibility"].value_counts(dropna=False))

In [ ]:
# Cell 6 — Cleaning and type normalization
NUMERIC_COLS = [
    "age", "monthly_salary", "years_of_employment", "monthly_rent",
    "family_size", "dependents", "school_fees", "college_fees",
    "travel_expenses", "groceries_utilities", "other_monthly_expenses",
    "current_emi_amount", "credit_score", "bank_balance",
    "emergency_fund", "requested_amount", "requested_tenure",
    "max_monthly_emi"
]

CATEGORICAL_COLS = [
    "gender", "marital_status", "education", "employment_type",
    "company_type", "house_type", "existing_loans", "emi_scenario",
    "emi_eligibility"
]

for col in NUMERIC_COLS:
    if col in df.columns:
        df[col] = (
            df[col].astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("₹", "", regex=False)
            .str.strip()
            .replace({"": np.nan, "nan": np.nan, "None": np.nan})
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

for col in CATEGORICAL_COLS:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

# Keep the original eligibility labels.
# The real dataset contains: Eligible, Not_Eligible, High_Risk.
# For binary eligibility classification, Eligible = 1 and every other outcome = 0.
df = df.dropna(subset=["emi_eligibility", "max_monthly_emi"]).drop_duplicates().reset_index(drop=True)

print("Cleaned shape:", df.shape)
print("\nEligibility labels:")
display(df["emi_eligibility"].value_counts(dropna=False))
display(df.head())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# OPTIONAL — Quick diagnostic
# Run this if another error appears. It tells us exactly where the problem is.
print("Pandas:", pd.__version__)
print("Scikit-learn:", __import__("sklearn").__version__)
print("XGBoost:", __import__("xgboost").__version__)
print("MLflow:", mlflow.__version__ if "mlflow" in globals() else "not imported yet")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

In [ ]:
# Cell 7 — Target distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["emi_eligibility"].value_counts().plot(kind="bar", ax=axes[0])
axes[0].set_title("EMI Eligibility Distribution")
axes[0].set_xlabel("Eligibility")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=0)

sns.histplot(df["max_monthly_emi"], bins=50, kde=True, ax=axes[1])
axes[1].set_title("Maximum Monthly EMI Distribution")
axes[1].set_xlabel("Max Monthly EMI")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 8 — Financial-variable EDA
eda_cols = [
    "monthly_salary", "current_emi_amount", "credit_score",
    "bank_balance", "emergency_fund", "requested_amount",
    "requested_tenure", "max_monthly_emi"
]
available = [c for c in eda_cols if c in df.columns]

df[available].describe().T

In [ ]:
# Cell 9 — Correlation heatmap
numeric_for_corr = df.select_dtypes(include=np.number)
corr = numeric_for_corr.corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Numeric Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## 4. Feature Engineering

We create financial indicators that are useful for lending decisions:

- **Debt-to-income ratio (DTI)**
- **Expense-to-income ratio**
- **Current EMI-to-income ratio**
- **Disposable income**
- **Requested amount-to-income ratio**
- **Bank balance-to-income ratio**
- **Emergency fund coverage**
- **Affordability ratio**
- **Credit score band**
- **A combined financial risk score**

Target columns are never used as input features, preventing direct target leakage.

In [ ]:
# Cell 10 — Feature engineering
def add_features(data):
    d = data.copy()

    income = d["monthly_salary"].clip(lower=1)
    total_expenses = (
        d["monthly_rent"].fillna(0)
        + d["school_fees"].fillna(0)
        + d["college_fees"].fillna(0)
        + d["travel_expenses"].fillna(0)
        + d["groceries_utilities"].fillna(0)
        + d["other_monthly_expenses"].fillna(0)
        + d["current_emi_amount"].fillna(0)
    )

    d["total_monthly_expenses"] = total_expenses
    d["disposable_income"] = d["monthly_salary"] - total_expenses
    d["dti_ratio"] = (d["current_emi_amount"] / income).clip(0, 5)
    d["expense_to_income_ratio"] = (total_expenses / income).clip(0, 10)
    d["emi_to_income_ratio"] = (d["current_emi_amount"] / income).clip(0, 5)
    d["requested_to_income_ratio"] = (d["requested_amount"] / income).clip(0, 100)
    d["bank_balance_to_income"] = (d["bank_balance"] / income).clip(0, 100)
    d["emergency_fund_months"] = (d["emergency_fund"] / income).clip(0, 120)
    d["affordable_emi_estimate"] = (d["disposable_income"] * 0.40).clip(lower=0)

    # A transparent rule-based risk indicator used only as an engineered input.
    credit_penalty = np.clip((700 - d["credit_score"]) / 200, 0, 3)
    d["financial_risk_score"] = (
        0.40 * d["dti_ratio"].clip(0, 1)
        + 0.25 * d["expense_to_income_ratio"].clip(0, 2)
        + 0.20 * credit_penalty
        + 0.15 * (1 / (1 + d["emergency_fund_months"]))
    )

    d["credit_score_band"] = pd.cut(
        d["credit_score"],
        bins=[0, 580, 670, 740, 850],
        labels=["Poor", "Fair", "Good", "Excellent"],
        include_lowest=True
    )

    return d

df_model = add_features(df)

print("Feature count before modeling:", df_model.shape[1])
display(df_model.head())

In [ ]:
# Cell 11 — EDA of engineered features
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_model.sample(min(20_000, len(df_model)), random_state=RANDOM_STATE),
    x="dti_ratio",
    y="credit_score",
    hue="emi_eligibility",
    alpha=0.45
)
plt.title("Credit Score vs DTI Ratio")
plt.tight_layout()
plt.show()

display(
    df_model.groupby("emi_eligibility")[
        ["credit_score", "dti_ratio", "expense_to_income_ratio", "disposable_income"]
    ].mean().round(2)
)

## 5. Prepare Classification and Regression Data

We use a stratified split for classification. The regression target is `max_monthly_emi`.

For practical Colab runtime, `TRAIN_SAMPLE = 150000` by default. The notebook still performs EDA on the complete dataset. Change it to `None` to train on all rows.

In [ ]:
# Cell 12 — Feature matrix and train/test split
TARGET_CLASS = "emi_eligibility"
TARGET_REG = "max_monthly_emi"

X_all = df_model.drop(columns=[TARGET_CLASS, TARGET_REG], errors="ignore")

# Binary business definition:
# Eligible -> 1
# Not_Eligible / High_Risk -> 0
y_class = (
    df_model[TARGET_CLASS]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("eligible")
    .astype(int)
)

y_reg = pd.to_numeric(df_model[TARGET_REG], errors="coerce")

valid_idx = y_reg.notna()
X_all = X_all.loc[valid_idx].reset_index(drop=True)
y_class = y_class.loc[valid_idx].reset_index(drop=True)
y_reg = y_reg.loc[valid_idx].reset_index(drop=True)

if TRAIN_SAMPLE is not None and TRAIN_SAMPLE < len(X_all):
    sample_idx = np.random.default_rng(RANDOM_STATE).choice(
        len(X_all), size=TRAIN_SAMPLE, replace=False
    )
    X_work = X_all.iloc[sample_idx].reset_index(drop=True)
    y_class_work = y_class.iloc[sample_idx].reset_index(drop=True)
    y_reg_work = y_reg.iloc[sample_idx].reset_index(drop=True)
else:
    X_work, y_class_work, y_reg_work = X_all, y_class, y_reg

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_work, y_class_work,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_class_work
)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_work, y_reg_work,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Classification train/test:", X_train_c.shape, X_test_c.shape)
print("Regression train/test:", X_train_r.shape, X_test_r.shape)
print("Classification labels:", y_class_work.value_counts().to_dict())

In [ ]:
# Cell 13 — Preprocessing pipelines
categorical_features = X_work.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()
numeric_features = [c for c in X_work.columns if c not in categorical_features]

numeric_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Sparse output is efficient for Logistic Regression, Random Forest and XGBoost.
categorical_sparse = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

# GradientBoostingClassifier/Regressor requires dense input.
categorical_dense = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor_scaled = ColumnTransformer([
    ("num", numeric_scaled, numeric_features),
    ("cat", categorical_sparse, categorical_features)
])

preprocessor_tree = ColumnTransformer([
    ("num", numeric_tree, numeric_features),
    ("cat", categorical_sparse, categorical_features)
])

preprocessor_dense = ColumnTransformer([
    ("num", numeric_tree, numeric_features),
    ("cat", categorical_dense, categorical_features)
])

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

## 6. Classification — EMI Eligibility

Models:
- Logistic Regression
- Random Forest
- XGBoost
- Gradient Boosting

Metrics:
- Accuracy
- Precision
- Recall
- F1
- ROC-AUC

In [ ]:
# Cell 14 — Train classification models
classification_models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor_scaled),
        ("model", LogisticRegression(max_iter=300, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor_tree),
        ("model", RandomForestClassifier(
            n_estimators=100, max_depth=18, n_jobs=-1,
            class_weight="balanced", random_state=RANDOM_STATE
        ))
    ]),
    "XGBoost": Pipeline([
        ("preprocessor", preprocessor_tree),
        ("model", XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.08,
            subsample=0.85, colsample_bytree=0.85,
            eval_metric="logloss", n_jobs=-1, random_state=RANDOM_STATE
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocessor", preprocessor_dense),
        ("model", GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.08, max_depth=4,
            random_state=RANDOM_STATE
        ))
    ])
}

classification_results = []
classification_fitted = {}

for name, model in classification_models.items():
    print(f"Training {name}...")
    model.fit(X_train_c, y_train_c)
    pred = model.predict(X_test_c)
    prob = model.predict_proba(X_test_c)[:, 1] if hasattr(model, "predict_proba") else None

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test_c, pred),
        "Precision": precision_score(y_test_c, pred, zero_division=0),
        "Recall": recall_score(y_test_c, pred, zero_division=0),
        "F1": f1_score(y_test_c, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test_c, prob) if prob is not None else np.nan
    }
    classification_results.append(metrics)
    classification_fitted[name] = model

classification_results_df = pd.DataFrame(classification_results).sort_values(
    "F1", ascending=False
).reset_index(drop=True)

display(classification_results_df)

In [ ]:
# Cell 15 — Classification diagnostics
best_class_name = classification_results_df.loc[0, "Model"]
best_class_model = classification_fitted[best_class_name]

best_class_pred = best_class_model.predict(X_test_c)

print("Best classification model:", best_class_name)
print(classification_report(
    y_test_c, best_class_pred,
    target_names=["Not Eligible", "Eligible"],
    zero_division=0
))

cm = confusion_matrix(y_test_c, best_class_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Not Eligible", "Eligible"],
            yticklabels=["Not Eligible", "Eligible"])
plt.title(f"Confusion Matrix — {best_class_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 7. Regression — Maximum Monthly EMI

Models:
- Linear Regression
- Random Forest
- XGBoost
- Gradient Boosting

Metrics:
- MAE
- RMSE
- R²

In [ ]:
# Cell 16 — Train regression models
regression_models = {
    "Linear Regression": Pipeline([
        ("preprocessor", preprocessor_scaled),
        ("model", LinearRegression())
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor_tree),
        ("model", RandomForestRegressor(
            n_estimators=100, max_depth=18, n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),
    "XGBoost": Pipeline([
        ("preprocessor", preprocessor_tree),
        ("model", XGBRegressor(
            n_estimators=200, max_depth=6, learning_rate=0.08,
            subsample=0.85, colsample_bytree=0.85,
            objective="reg:squarederror", n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocessor", preprocessor_dense),
        ("model", GradientBoostingRegressor(
            n_estimators=100, learning_rate=0.08, max_depth=4,
            random_state=RANDOM_STATE
        ))
    ])
}

regression_results = []
regression_fitted = {}

for name, model in regression_models.items():
    print(f"Training {name}...")
    model.fit(X_train_r, y_train_r)
    pred = model.predict(X_test_r)

    metrics = {
        "Model": name,
        "MAE": mean_absolute_error(y_test_r, pred),
        "RMSE": mean_squared_error(y_test_r, pred) ** 0.5,
        "R2": r2_score(y_test_r, pred)
    }
    regression_results.append(metrics)
    regression_fitted[name] = model

regression_results_df = pd.DataFrame(regression_results).sort_values(
    "R2", ascending=False
).reset_index(drop=True)

display(regression_results_df)

In [ ]:
# Cell 17 — Regression diagnostics
best_reg_name = regression_results_df.loc[0, "Model"]
best_reg_model = regression_fitted[best_reg_name]

best_reg_pred = best_reg_model.predict(X_test_r)

print("Best regression model:", best_reg_name)
print("MAE :", mean_absolute_error(y_test_r, best_reg_pred))
print("RMSE:", mean_squared_error(y_test_r, best_reg_pred) ** 0.5)
print("R²  :", r2_score(y_test_r, best_reg_pred))

plt.figure(figsize=(8, 6))
plt.scatter(y_test_r, best_reg_pred, alpha=0.25)
lims = [
    min(float(y_test_r.min()), float(best_reg_pred.min())),
    max(float(y_test_r.max()), float(best_reg_pred.max()))
]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Actual Max EMI")
plt.ylabel("Predicted Max EMI")
plt.title(f"Actual vs Predicted — {best_reg_name}")
plt.tight_layout()
plt.show()

## 8. MLflow Experiment Tracking + Model Registry

This section logs:
- model parameters
- evaluation metrics
- the trained sklearn pipeline as an MLflow model

A local SQLite backend is used so the notebook works in Colab without requiring a separate MLflow server.

The registered model names are:
- `EMIPredict_Eligibility`
- `EMIPredict_MaxEMI`

In [ ]:
# Cell 18 — MLflow setup
import mlflow
import mlflow.sklearn

MLFLOW_DIR = BASE_DIR / "mlruns"
MLFLOW_DB = BASE_DIR / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
mlflow.set_experiment("EMIPredict_AI")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

In [ ]:
# Cell 19 — Log classification experiments
for row in classification_results:
    name = row["Model"]
    model = classification_fitted[name]
    with mlflow.start_run(run_name=f"classification_{name.replace(' ', '_')}"):
        mlflow.log_params({
            "task": "classification",
            "model": name,
            "train_rows": len(X_train_c),
            "test_rows": len(X_test_c)
        })
        mlflow.log_metrics({
            k: float(v) for k, v in row.items()
            if k not in ["Model"] and pd.notna(v)
        })
        mlflow.sklearn.log_model(
            model,
            artifact_path="model",
            registered_model_name="EMIPredict_Eligibility"
        )

print("Classification experiments logged.")

In [ ]:
# Cell 20 — Log regression experiments
for row in regression_results:
    name = row["Model"]
    model = regression_fitted[name]
    with mlflow.start_run(run_name=f"regression_{name.replace(' ', '_')}"):
        mlflow.log_params({
            "task": "regression",
            "model": name,
            "train_rows": len(X_train_r),
            "test_rows": len(X_test_r)
        })
        mlflow.log_metrics({
            k: float(v) for k, v in row.items()
            if k not in ["Model"] and pd.notna(v)
        })
        mlflow.sklearn.log_model(
            model,
            artifact_path="model",
            registered_model_name="EMIPredict_MaxEMI"
        )

print("Regression experiments logged.")

## 9. Best-Model Selection & Export

The best classification model is selected by **F1 score**.  
The best regression model is selected by **R²**.

The exported pipelines include preprocessing + feature transformation + model, making them directly reusable by a Streamlit application.

In [ ]:
# Cell 21 — Export best models and evaluation reports
best_class_path = MODELS_DIR / "best_classification_model.joblib"
best_reg_path = MODELS_DIR / "best_regression_model.joblib"

joblib.dump(best_class_model, best_class_path)
joblib.dump(best_reg_model, best_reg_path)

classification_results_df.to_csv(REPORTS_DIR / "classification_results.csv", index=False)
regression_results_df.to_csv(REPORTS_DIR / "regression_results.csv", index=False)

# Save the feature list used by the Streamlit app.
feature_metadata = {
    "feature_columns": X_work.columns.tolist(),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "classification_target": TARGET_CLASS,
    "regression_target": TARGET_REG,
    "best_classification_model": best_class_name,
    "best_regression_model": best_reg_name
}
joblib.dump(feature_metadata, ARTIFACTS_DIR / "feature_metadata.joblib")

print("Saved:")
print(best_class_path)
print(best_reg_path)
print(REPORTS_DIR / "classification_results.csv")
print(REPORTS_DIR / "regression_results.csv")
print(ARTIFACTS_DIR / "feature_metadata.joblib")

In [ ]:
# Cell 22 — Verify exported models
loaded_class_model = joblib.load(best_class_path)
loaded_reg_model = joblib.load(best_reg_path)

sample_input = X_test_c.head(5)

class_check = loaded_class_model.predict(sample_input)
reg_check = loaded_reg_model.predict(sample_input)

print("Classification predictions:", class_check)
print("Regression predictions:", np.round(reg_check, 2))
print("\nModel files exist:")
print(best_class_path.exists(), best_reg_path.exists())

## 10. Streamlit Deployment Test

The exported models can be used by a Streamlit app like this:

```python
import joblib

classification_model = joblib.load("models/best_classification_model.joblib")
regression_model = joblib.load("models/best_regression_model.joblib")

eligibility = classification_model.predict(input_df)[0]
max_emi = regression_model.predict(input_df)[0]
```

Because feature engineering and preprocessing are inside the pipelines, the app should provide the **raw input columns** expected by the models.

### Suggested Streamlit outputs
- EMI eligibility: **Eligible / Not Eligible**
- Maximum recommended monthly EMI
- Credit/risk summary
- Key applicant financial indicators
- Optional probability of eligibility

> Never expose or rely on sensitive banking information beyond what is necessary for the application, and validate all user inputs before prediction.

In [ ]:
# Cell 23 — Create a deployment bundle
import shutil

deployment_dir = BASE_DIR / "deployment"
deployment_dir.mkdir(exist_ok=True)

for src in [best_class_path, best_reg_path, ARTIFACTS_DIR / "feature_metadata.joblib"]:
    shutil.copy2(src, deployment_dir / src.name)

print("Deployment bundle:")
for p in deployment_dir.iterdir():
    print(" -", p)

# Final Project Summary

### EMIPredict AI
**Input:** applicant financial, employment, household, credit, loan and expense information.

### Classification output
Predicts whether the applicant is:
- **Eligible**
- **Not Eligible**

### Regression output
Predicts the applicant's **maximum recommended monthly EMI**.

### ML pipeline
**Data → Cleaning → EDA → Feature Engineering → Preprocessing → Classification/Regression → Evaluation → MLflow → Best Model → `.joblib` → Streamlit**

### Deliverables
- `models/best_classification_model.joblib`
- `models/best_regression_model.joblib`
- `reports/classification_results.csv`
- `reports/regression_results.csv`
- `artifacts/feature_metadata.joblib`
- MLflow tracking database and runs

### Recommended GitHub structure

```text
EMIPredict-AI/
├── data/
│   └── emi_prediction_dataset.csv
├── models/
│   ├── best_classification_model.joblib
│   └── best_regression_model.joblib
├── notebooks/
│   └── EMIPredict_AI_Colab_Notebook.ipynb
├── reports/
│   ├── classification_results.csv
│   └── regression_results.csv
├── app.py
├── requirements.txt
└── README.md
```